# 01 -- Datenexploration und Variablenauswahl

Dieses Notebook deckt die Tasks 1-4 des praktischen Analyseplans ab:

| Task | Beschreibung |
|------|-------------|
| **1** | Zielvariablen festlegen (HDI & Happiness Score) |
| **2** | Betrachtungszeitraum bestimmen |
| **3** | Betrachtete Laender bestimmen |
| **4** | Makrooekonomische Indikatoren auswaehlen |

**Datenquellen:**
- WISE Database (Liu et al., 2024): >1 Mio. Datenpunkte, 244 Metriken, 218 Laender
- Jones & Klenow (2016): lambda-Werte fuer 152 Laender (Referenzjahr 2007) -- wird in Notebook 04 verwendet

**Reproduzierbarkeit:** Alle Entscheidungen (Zeitraum, Laender, Indikatoren) werden in diesem Notebook
a priori dokumentiert und begruendet, bevor die Analyse stattfindet.

## 0 -- Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Reproduzierbarkeit
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Pfade (relativ zum /code-Verzeichnis)
ROOT      = Path('..') / 'Pulled Databases'
WISE_PATH = ROOT / 'WISE_Database' / 'WISE_Database' / 'Data' / 'WISE_Database' / 'WISE_Database.xlsx'
JK_PATH   = ROOT / 'Jones_and_Klenow_Database' / 'BeyondGDP500.xls'

assert WISE_PATH.exists(), f'WISE-Datei nicht gefunden: {WISE_PATH}'
assert JK_PATH.exists(),   f'Jones/Klenow-Datei nicht gefunden: {JK_PATH}'

## 1 -- WISE Metadaten laden

Die `Metrics Info`-Sheet enthält alle Metadaten zu den 244 Metriken der WISE-Datenbank,
inkl. Kategorisierung (Wellbeing/Inclusion/Sustainability/Economy), Zeitraum und Laenderabdeckung.

In [ ]:
metrics_info = pd.read_excel(WISE_PATH, sheet_name='Metrics Info')

print(f'Anzahl Metriken: {len(metrics_info)}')
print(f'Spalten: {list(metrics_info.columns)}')
metrics_info.head(3)

## Task 1 -- Zielvariablen: HDI & Happiness Score

Die Arbeit verwendet zwei komplementäre Zielvariablen, die zusammen die objektive
und subjektive Seite des Wohlergehens abdecken (vgl. CMEPSP 2009)

### Übersicht aller HDI-verwandten Variablen in der WISE-Datenbank

Die folgende Tabelle listet alle Variablen, deren Akronym 'HDI' enthaelt oder
deren vollstaendiger Name 'Human Development Index' enthaelt.
Spalten: Variablenname (Akronym), Anzahl Laender, Zeitraum, Beschreibung.
Fehlende Beschreibungen ('—') bedeuten, dass die WISE-Datenbank fuer diese
Variable keinen Eintrag in der Spalte 'Metric Description' hat.

In [ ]:
# Tabelle 1: Alle HDI-verwandten Variablen (direkt aus WISE Metrics Info)
hdi_vars = metrics_info[
    metrics_info['Acronym'].str.contains('HDI', na=False) |
    metrics_info['Metric Full Name'].str.contains('Human Development Index', case=False, na=False)
][['Metric Full Name', 'Acronym', 'Available Country Count', 'Start Year', 'End Year', 'Metric Description']].copy()

hdi_vars['Zeitraum'] = (
    hdi_vars['Start Year'].astype(int).astype(str) + '-' +
    hdi_vars['End Year'].astype(int).astype(str)
)
hdi_vars['Beschreibung'] = hdi_vars['Metric Description'].fillna('--')

hdi_display = hdi_vars[['Metric Full Name', 'Acronym', 'Available Country Count', 'Zeitraum', 'Beschreibung']].rename(columns={
    'Metric Full Name':        'Name',
    'Acronym':                 'Variablenname',
    'Available Country Count': 'Anzahl Laender',
}).reset_index(drop=True)

print(f'Anzahl HDI-verwandter Variablen in WISE: {len(hdi_display)}')
pd.set_option('display.max_colwidth', 300)
display(hdi_display)

### Übersicht aller Life-Satisfaction-Variablen in der WISE-Datenbank

Die folgende Tabelle listet alle Variablen mit Theme 'Subjective Wellbeing'
oder Akronym beginnend mit 'WHR'.
Spalten: Variablenname (Akronym), Anzahl Laender, Zeitraum, Beschreibung.

In [ ]:
# Tabelle 2: Alle LS / Subjective-Wellbeing-Variablen (direkt aus WISE Metrics Info)
ls_vars = metrics_info[
    (
        (metrics_info['Theme'] == 'Subjective Wellbeing') |
        (metrics_info['Acronym'].str.startswith('WHR', na=False))
    ) &
    (~metrics_info['Acronym'].str.startswith('FAN-DEF-SF-SS', na=False))
][['Metric Full Name', 'Acronym', 'Available Country Count', 'Start Year', 'End Year', 'Metric Description']].copy()

ls_vars['Zeitraum'] = (
    ls_vars['Start Year'].astype(int).astype(str) + '-' +
    ls_vars['End Year'].astype(int).astype(str)
)
ls_vars['Beschreibung'] = ls_vars['Metric Description'].fillna('--')

ls_display = ls_vars[['Metric Full Name', 'Acronym', 'Available Country Count', 'Zeitraum', 'Beschreibung']].rename(columns={
    'Metric Full Name':        'Name',
    'Acronym':                 'Variablenname',
    'Available Country Count': 'Anzahl Laender',
}).reset_index(drop=True)

print(f'Anzahl LS-verwandter Variablen in WISE: {len(ls_display)}')
display(ls_display)

In [ ]:
# Beide Tabellen in eine Excel-Datei mit zwei Sheets speichern
OUTPUT_FILE = 'possible_target_vars.xlsx'

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    hdi_display.to_excel(writer, sheet_name='HDI_Variables',  index=False)
    ls_display.to_excel( writer, sheet_name='LS_Variables',   index=False)

print(f'Gespeichert: {OUTPUT_FILE}')
print(f'  Sheet "HDI_Variables": {len(hdi_display)} Variablen')
print(f'  Sheet "LS_Variables":  {len(ls_display)} Variablen')

### Variablen für kommende Analyse

| Variable | Akronym | Art | Quelle |
|----------|---------|-----|--------|
| Human Development Index | `UNDP-HDI` | Objektiv | UNDP |
| Life Satisfaction (Cantril-Ladder) | `WHR-LS` | Subjektiv | World Happiness Report / Gallup |

In [20]:
TARGET_ACRONYMS = ['UNDP-HDI', 'WHR-LS']

target_meta = metrics_info[
    metrics_info['Acronym'].isin(TARGET_ACRONYMS)
][[
    'Metric Full Name', 'Acronym', 'Unit', 'Tier',
    'Available Country Count', 'Start Year', 'End Year',
    'Wellbeing', 'Subjective', 'Index', 'Theme'
]].reset_index(drop=True)

print('=== Zielvariablen in der WISE-Datenbank ===')
display(target_meta)

hdi_meta = target_meta[target_meta['Acronym'] == 'UNDP-HDI'].iloc[0]
whr_meta = target_meta[target_meta['Acronym'] == 'WHR-LS'].iloc[0]

print(f"HDI:             {int(hdi_meta['Available Country Count'])} Laender, "
      f"{int(hdi_meta['Start Year'])}-{int(hdi_meta['End Year'])}")
print(f"Happiness Score: {int(whr_meta['Available Country Count'])} Laender, "
      f"{int(whr_meta['Start Year'])}-{int(whr_meta['End Year'])}")

=== Zielvariablen in der WISE-Datenbank ===


,Metric Full Name,Acronym,Unit,Tier,Available Country Count,Start Year,End Year,Wellbeing,Subjective,Index,Theme
0,Human Development Index,UNDP-HDI,Index (0-1),1st,193,1990,2022,X,NaN,X,Summary
1,Life Satisfaction,WHR-LS,Index(0-10),1st,164,2006,2023,X,X,X,Subjective Wellbeing


HDI:             193 Laender, 1990-2022
Happiness Score: 164 Laender, 2006-2023


## Task 2 -- Betrachtungszeitraum

Der Betrachtungszeitraum wird durch den engsten gemeinsamen Überlappungsbereich
beider Zielvariablen mit ausreichender Datenqualität bestimmt. Die Entscheidung erfolgt auf Basis der
Datenverfügbarkeit und wird durch die empirische Länderabdeckung je Jahr validiert.

In [ ]:
# Nur die beiden Zielvariablen aus C Data laden
print('Lade Zielvariablen aus C Data für Datenqualitätsanalyse ...')
quality_raw = pd.read_excel(
    WISE_PATH,
    sheet_name='C Data',
    usecols=['ISO3', 'Acronym', 'Year', 'Value']
)
quality_raw = quality_raw[quality_raw['Acronym'].isin(TARGET_ACRONYMS)].copy()
quality_raw['Year'] = quality_raw['Year'].astype(int)
print(f'Zeilen geladen: {len(quality_raw):,}')

Lade Zielvariablen aus C Data fuer Datenqualitaetsanalyse ...
Zeilen geladen: 8,171


In [24]:
# Für jede Variable: Anzahl eindeutiger Lnder insgesamt (Nenner)
total_countries = (
    quality_raw[quality_raw['Value'].notna()]
    .groupby('Acronym')['ISO3']
    .nunique()
    .rename('total_countries')
)

# Für jedes (Variable, Jahr): Anzahl Länder mit Datenpunkt (Zähler)
yearly_counts = (
    quality_raw[quality_raw['Value'].notna()]
    .groupby(['Acronym', 'Year'])['ISO3']
    .nunique()
    .reset_index(name='n_countries')
)

# Coverage in % berechnen
yearly_counts = yearly_counts.merge(total_countries, on='Acronym')
yearly_counts['coverage_pct'] = (
    yearly_counts['n_countries'] / yearly_counts['total_countries'] * 100
).round(1)

# Pivot für Anzeige
quality_table = yearly_counts.pivot(
    index='Year', columns='Acronym', values='coverage_pct'
).rename(columns={'UNDP-HDI': 'HDI_coverage_pct', 'WHR-LS': 'LS_coverage_pct'})

print('=== Datenabdeckung je Jahr (% der Länder mit Datenpunkt) ===')
print(f'Nenner HDI: {int(total_countries["UNDP-HDI"])} Länder insgesamt')
print(f'Nenner LS:  {int(total_countries["WHR-LS"])} Länder insgesamt')
print()
display(quality_table.T.style.format('{:.1f}%', na_rep='--').background_gradient(
    cmap='RdYlGn', vmin=0, vmax=100, axis=1
))

=== Datenabdeckung je Jahr (% der Länder mit Datenpunkt) ===
Nenner HDI: 193 Länder insgesamt
Nenner LS:  164 Länder insgesamt



Year,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
Acronym,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HDI_coverage_pct,73.6%,73.6%,73.6%,73.6%,73.6%,79.3%,79.3%,79.3%,79.3%,81.9%,91.2%,91.2%,91.7%,92.7%,93.8%,97.4%,97.4%,97.9%,97.9%,97.9%,99.0%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,99.5%,100.0%,--
LS_coverage_pct,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,16.5%,54.3%,62.2%,67.1%,69.5%,75.6%,89.0%,86.0%,82.9%,87.8%,86.6%,86.0%,89.6%,86.0%,87.2%,70.7%,74.4%,85.4%,84.1%


### ENTSCHEIDUNG TASK 2: Betrachtungszeitraum
 Begründung:
   - HDI verfügbar ab 1990; Happiness Score erst ab 2006.
   - Engster gemeinsamer Überlappungsbereich: 2006-2022 (17 Jahre).
   - Datenqualität für Happiness Score in 2006 jedoch nicht ausreichend.
   - Unabhängig von Analyseergebnissen festgelegt.

In [30]:
YEAR_START = 2007
YEAR_END   = 2022

target_period = target_wide[
    (target_wide['Year'] >= YEAR_START) &
    (target_wide['Year'] <= YEAR_END)
].copy()

print(f'Gewählter Zeitraum: {YEAR_START}-{YEAR_END} ({YEAR_END - YEAR_START + 1} Jahre)')
print(f'Länder-Jahr-Beobachtungen im Zeitraum: {len(target_period):,}')
print(f'Länder mit mind. 1 HDI-Beobachtung:             '
      f'{target_period[target_period["HDI"].notna()]["ISO3"].nunique()}')
print(f'Länder mit mind. 1 Happiness-Score-Beobachtung: '
      f'{target_period[target_period["Happiness_Score"].notna()]["ISO3"].nunique()}')

Gewählter Zeitraum: 2007-2022 (16 Jahre)
Länder-Jahr-Beobachtungen im Zeitraum: 3,100
Länder mit mind. 1 HDI-Beobachtung:             193
Länder mit mind. 1 Happiness-Score-Beobachtung: 163
